# KTPM-BTL: một Admin, bác sĩ và khách hàng tăng theo tải
Chọn CPU None và Internet On. Notebook chạy một notebook con bằng nbclient; file executed-fixed-admin.ipynb trong ZIP là notebook thực thi thật, có execution_count và outputs.

In [ ]:
from pathlib import Path
import sys,subprocess,json
ROOT=Path('/kaggle/working/KTPM-BTL')
SOURCE_REF='main'
assert Path('/kaggle/working').is_dir()
if not (ROOT/'.git').exists():
    subprocess.run(['git','clone','--branch',SOURCE_REF,'--single-branch','https://github.com/FlowBoat123/KTPM-BTL.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',SOURCE_REF],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
COMMIT=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
subprocess.run([sys.executable,'-m','pip','install','-q','nbformat==5.10.4','nbclient==0.10.2'],check=True)
print('Code to measure:',COMMIT)


In [ ]:
import nbformat
from nbclient import NotebookClient
CHILD_CODES=["from pathlib import Path\nimport os, sys, subprocess, json, datetime, shutil\nassert Path('/kaggle/working').is_dir(), 'Run on Kaggle CPU'\nROOT=Path('/kaggle/working/KTPM-BTL')\nCOMMIT='__COMMIT__'\nREPO='https://github.com/FlowBoat123/KTPM-BTL.git'\nif not (ROOT/'.git').exists():\n    subprocess.run(['git','clone','--no-checkout',REPO,str(ROOT)],check=True)\nsubprocess.run(['git','-C',str(ROOT),'fetch','origin',COMMIT],check=True)\nsubprocess.run(['git','-C',str(ROOT),'checkout','--detach',COMMIT],check=True)\nassert subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()==COMMIT\ngpu=subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True) if shutil.which('nvidia-smi') else None\nassert gpu is None or gpu.returncode!=0 or not gpu.stdout.strip(), 'Choose Accelerator None'\nprint('Source commit:', COMMIT)\n","JAVA_HOME=Path('/usr/lib/jvm/java-17-openjdk-amd64')\nif not (JAVA_HOME/'bin/java').exists():\n    subprocess.run(['apt-get','update','-qq'],check=True,stdout=subprocess.DEVNULL)\n    subprocess.run(['apt-get','install','-y','openjdk-17-jdk-headless'],check=True,stdout=subprocess.DEVNULL)\nos.environ['JAVA_HOME']=str(JAVA_HOME)\nos.environ['PATH']=str(JAVA_HOME/'bin')+os.pathsep+os.environ['PATH']\nsubprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-load.txt')],check=True)\n(ROOT/'mvnw').chmod(0o755)\nwith Path('/kaggle/working/build.log').open('w') as log:\n    build=subprocess.run([str(ROOT/'mvnw'),'-B','verify'],cwd=ROOT,stdout=log,stderr=subprocess.STDOUT)\nprint('\\n'.join(Path('/kaggle/working/build.log').read_text().splitlines()[-20:]))\nassert build.returncode==0, 'Build/test failed'\n","SCENARIOS=[(100,5),(200,10),(500,20)]  # customers, distinct doctors; exactly one admin\nDURATION=120\nWARMUP=15\nSPAWN_RATE=50\nRUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')\nOUTPUT_ROOT=Path('/kaggle/working')/('fixed-admin-'+RUN_ID)\nOUTPUT_ROOT.mkdir()\nRESULTS=[]\nfor customers,doctors in SCENARIOS:\n    output=OUTPUT_ROOT/f'customers-{customers}-doctors-{doctors}'\n    print(f'Start: admin=1, doctors={doctors}, customers={customers}',flush=True)\n    run=subprocess.run([sys.executable,str(ROOT/'scripts/run_load_test.py'),\n        '--environment','kaggle','--customers',str(customers),'--doctors',str(doctors),\n        '--spawn-rate',str(SPAWN_RATE),'--warmup',str(WARMUP),'--duration',str(DURATION),\n        '--output',str(output)],cwd=ROOT)\n    meta=json.loads((output/'metadata.json').read_text())\n    summary=json.loads((output/'summary.json').read_text()) if (output/'summary.json').exists() else {}\n    phases=json.loads((output/'phases.json').read_text()) if (output/'phases.json').exists() else {}\n    expected={'AdminUser':1,'DoctorUser':doctors,'CustomerUser':customers}\n    assert meta['commit']==COMMIT and meta['working_tree']=='', 'Source changed'\n    assert phases.get('observed_counts_at_end')==expected, 'Wrong role populations'\n    row={'customers':customers,'doctors':doctors,'admins':1,'total_users':1+customers+doctors,\n         'exit_code':run.returncode,**summary}\n    RESULTS.append(row)\n    (OUTPUT_ROOT/'comparison.json').write_text(json.dumps(RESULTS,indent=2))\n    print(json.dumps(row),flush=True)\n    assert 'requests' in summary, 'Run failed; inspect logs'\nPath('/kaggle/working/latest-fixed-admin-output.txt').write_text(str(OUTPUT_ROOT))\n","from IPython.display import display, Markdown, Image\nimport csv\nimport matplotlib.pyplot as plt\nwith (OUTPUT_ROOT/'comparison.csv').open('w',newline='') as stream:\n    writer=csv.DictWriter(stream,fieldnames=list(RESULTS[0]));writer.writeheader();writer.writerows(RESULTS)\nlines=['# Kaggle CPU: one admin, distinct doctors', '', 'Source commit: '+COMMIT,\n       'Same sessions across login/warm-up/measurement. Warm-up 15s AFTER every session is ready; measure 120s plus draining in-flight requests. See phases.json for actual elapsed time. Fresh backend/DB per scenario.',\n       '', '| Customers | Doctors | Admin | Total | Requests | Failures | RPS | Mean ms | P95 ms |',\n       '| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |']\nfor r in RESULTS:\n    lines.append('| {customers} | {doctors} | {admins} | {total_users} | {requests} | {failures} | {rps:.2f} | {mean_ms:.2f} | {p95_ms:.0f} |'.format(**r))\nlines+=['', 'Customers read/book/look up; 20% of new bookings cancel immediately. One admin reads and assigns pending bookings. Distinct doctors read/confirm/complete their own cases. Think time 1–2.5s per task.',\n        '', 'Login/startup metrics are in setup_stats.csv; measurement excludes login/warm-up. Backend and Locust share CPU; not maximum capacity. One admin may leave a growing pending queue. Historical 7/2/1 results are a different workload, not a before/after optimization comparison.']\n(OUTPUT_ROOT/'REPORT.md').write_text('\\n'.join(lines))\ndisplay(Markdown('\\n'.join(lines)))\nfig,axes=plt.subplots(1,2,figsize=(10,3.5))\nfor ax,key,label in zip(axes,['rps','p95_ms'],['Requests / second','P95 (ms)']):\n    ax.plot([r['customers'] for r in RESULTS],[r[key] for r in RESULTS],marker='o')\n    ax.set_xlabel('Customer sessions (admin=1; doctors=5/10/20)');ax.set_ylabel(label);ax.grid(alpha=.3)\nfig.tight_layout();fig.savefig(OUTPUT_ROOT/'scaling.png',dpi=160);plt.close(fig)\ndisplay(Image(filename=str(OUTPUT_ROOT/'scaling.png')))\n"]
child=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# KTPM-BTL: executed Kaggle CPU benchmark\nSource: '+COMMIT)]+[nbformat.v4.new_code_cell(c.replace('__COMMIT__',COMMIT)) for c in CHILD_CODES])
child.metadata.kernelspec={'name':'python3','display_name':'Python 3','language':'python'}
EXECUTED=Path('/kaggle/working/executed-fixed-admin.ipynb')
import threading
progress_stop=threading.Event()
def progress():
    while not progress_stop.wait(30):
        files=sorted(Path('/kaggle/working').glob('fixed-admin-*/comparison.json'))
        if files:
            rows=json.loads(files[-1].read_text())
            print('Completed scenarios:',len(rows),'latest:',json.dumps(rows[-1]),flush=True)
        else:
            print('Build / first scenario in progress',flush=True)
thread=threading.Thread(target=progress,daemon=True)
thread.start()
try:
    NotebookClient(child,timeout=1200,kernel_name='python3',resources={'metadata':{'path':str(ROOT)}}).execute()
finally:
    progress_stop.set()
    thread.join(timeout=2)
    nbformat.write(child,str(EXECUTED))
print('Executed notebook saved:',EXECUTED)


In [ ]:
from IPython.display import display,Markdown,Image,FileLink
import zipfile
OUTPUT_ROOT=Path('/kaggle/working/latest-fixed-admin-output.txt').read_text().strip()
OUTPUT_ROOT=Path(OUTPUT_ROOT)
display(Markdown((OUTPUT_ROOT/'REPORT.md').read_text()))
display(Image(filename=str(OUTPUT_ROOT/'scaling.png')))
BUNDLE=OUTPUT_ROOT.with_suffix('.zip')
with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as archive:
    for file in OUTPUT_ROOT.rglob('*'):
        if file.is_file() and file.suffix in {'.json','.csv','.html','.md','.png','.log'}:
            archive.write(file,file.relative_to(OUTPUT_ROOT))
    archive.write(EXECUTED,'executed-fixed-admin.ipynb')
    archive.write('/kaggle/working/build.log','build.log')
display(FileLink(str(BUNDLE)))
print('Archive:',BUNDLE)
